# Data Collection and Pre-Processing Lab

This notebook demonstrates an end-to-end data engineering workflow using an e-commerce sales dataset.

The workflow includes data ingestion, data structures, profiling, cleaning, transformation, feature engineering, aggregation, serialization, and reflection.

## Step 1 — Hello, Data!

The first step is to load the raw e-commerce CSV file without modifying the source data. The first three rows are displayed to verify that the dataset was loaded correctly.

In [70]:
from pathlib import Path
import sys
import pandas as pd

# Find the project root
PROJECT_ROOT = Path.cwd()

# If the notebook is running from the notebooks folder,
# move one level up to the project root.
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Add the project root to Python's import path
sys.path.append(str(PROJECT_ROOT))

# Define important folders
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"

# Primary dataset path
RAW_DATA_PATH = DATA_DIR / "retail_sales_ontario_synthetic.csv"

# Load the raw CSV
sales_data = pd.read_csv(
    RAW_DATA_PATH,
    low_memory=False
)

# Display the first 3 rows
sales_data.head(3)

,order_id,date,customer_id,product_id,product,product_category,price,quantity,coupon_code,discount_pct,payment_method,shipping_city,shipping_province,sales_amount
0,ON100762,03/01/2025,C11886,P2378,Lip Balm,Beauty,63.41,6.0,NO_COUPON,0,Credit Card,London,ON,380.46
1,ON100888,03/01/2025,C10858,P1339,Air Fryer,Home & Kitchen,55.39,5.0,WELCOME5,5,Interac,Guelph,ON,263.10
2,ON100375,04/01/2025,C12057,P8749,Coffee Maker,Home & Kitchen,81.91,3.0,NO_COUPON,0,Credit Card,Hamilton,ON,245.73


In [71]:
sales_data.columns.tolist()

['order_id',
 'date',
 'customer_id',
 'product_id',
 'product',
 'product_category',
 'price',
 'quantity',
 'coupon_code',
 'discount_pct',
 'payment_method',
 'shipping_city',
 'shipping_province',
 'sales_amount']

## Step 2 — Pick the Right Container

A dictionary is appropriate for a sales record because it stores related values using meaningful field names such as `customer_id`, `product`, and `price`, and it is easy to update. A set is useful when we only need unique values, such as unique shipping cities, while a namedtuple would be better for fixed, immutable records.

## Step 3 — Implement Functions and Data Structure

The `LoadSales` class in `src/Load_data.py` uses object-oriented programming to load and clean the sales data. The loaded DataFrame is also converted into a list of dictionaries so that each transaction is represented as a Python data structure.

In [72]:
from src.Load_data import LoadSales

# Create an object from our class
loader = LoadSales(RAW_DATA_PATH)

# Load the sales data
sales_data = loader.getSales()

# Convert the DataFrame into a list of dictionaries
sales_records = sales_data.to_dict(orient="records")

# Check the resulting data structure
print("Data structure type:", type(sales_records))
print("Number of records:", len(sales_records))

# Display the first record
sales_records[0]

Data structure type: <class 'list'>
Number of records: 1020


{'order_id': 'ON100762',
 'date': '03/01/2025',
 'customer_id': 'C11886',
 'product_id': 'P2378',
 'product': 'Lip Balm',
 'product_category': 'Beauty',
 'price': 63.41,
 'quantity': 6.0,
 'coupon_code': 'NO_COUPON',
 'discount_pct': 0,
 'payment_method': 'Credit Card',
 'shipping_city': 'London',
 'shipping_province': 'ON',
 'sales_amount': 380.46}

## Step 4 — Bulk Loaded

A DataFrame is useful for bulk data processing, while dictionaries are useful for lookup operations. The following example creates a dictionary mapping each customer ID to a shipping city.

In [73]:
customer_city_map = (
    sales_data[
        ["customer_id", "shipping_city"]
    ]
    .dropna()
    .drop_duplicates("customer_id")
    .set_index("customer_id")["shipping_city"]
    .to_dict()
)

# Display a few dictionary entries
list(customer_city_map.items())[:5]

[('C11886', 'London'),
 ('C10858', 'Guelph'),
 ('C12057', 'Hamilton'),
 ('C10823', 'Vaughan'),
 ('C10951', 'Scarborough')]

## Step 5 — Quick Profiling

Before cleaning the dataset, we need a quick profile. This includes the minimum, mean, and maximum product price and the number of unique shipping cities. A Python `set` is used to identify unique cities.

In [74]:
# Make sure price is numeric for profiling
sales_data["price"] = pd.to_numeric(
    sales_data["price"],
    errors="coerce"
)

# Price statistics
minimum_price = sales_data["price"].min()
average_price = sales_data["price"].mean()
maximum_price = sales_data["price"].max()

# Use a set to count unique cities
unique_cities = set(
    sales_data["shipping_city"]
    .dropna()
    .astype(str)
    .str.strip()
)

print(f"Minimum price: ${minimum_price:.2f}")
print(f"Mean price: ${average_price:.2f}")
print(f"Maximum price: ${maximum_price:.2f}")
print(f"Unique shipping cities: {len(unique_cities)}")

Minimum price: $3.10
Mean price: $57.68
Maximum price: $257.97
Unique shipping cities: 15


## Step 6 — Spot the Grime

The raw data is checked for common data-quality problems before cleaning. The checks include missing required values, duplicate rows, invalid dates, non-positive prices or quantities, and inconsistent text formatting such as extra whitespace.

In [75]:
# Check for common data-quality problems

invalid_dates = (
    pd.to_datetime(
        sales_data["date"],
        errors="coerce"
    ).isna().sum()
)

missing_required = (
    sales_data[
        [
            "date",
            "customer_id",
            "product",
            "price",
            "quantity",
            "shipping_city"
        ]
    ]
    .isna()
    .any(axis=1)
    .sum()
)

duplicate_rows = sales_data.duplicated().sum()

non_positive_price = (
    pd.to_numeric(
        sales_data["price"],
        errors="coerce"
    ) <= 0
).sum()

non_positive_quantity = (
    pd.to_numeric(
        sales_data["quantity"],
        errors="coerce"
    ) <= 0
).sum()

messy_city_names = (
    sales_data["shipping_city"]
    .dropna()
    .astype(str)
    .apply(lambda x: x != x.strip())
    .sum()
)

dirty_report = {
    "Missing required values": int(missing_required),
    "Invalid dates": int(invalid_dates),
    "Duplicate rows": int(duplicate_rows),
    "Non-positive prices": int(non_positive_price),
    "Non-positive quantities": int(non_positive_quantity),
    "Cities with extra whitespace": int(messy_city_names)
}

pd.Series(
    dirty_report,
    name="Number of affected rows"
)

Missing required values           4
Invalid dates                   635
Duplicate rows                    0
Non-positive prices               0
Non-positive quantities           0
Cities with extra whitespace      0
Name: Number of affected rows, dtype: int64

In [76]:
# Show examples of rows with missing required values
sales_data[
    sales_data[
        [
            "date",
            "customer_id",
            "product",
            "price",
            "quantity",
            "shipping_city"
        ]
    ]
    .isna()
    .any(axis=1)
].head(5)

,order_id,date,customer_id,product_id,product,product_category,price,quantity,coupon_code,discount_pct,payment_method,shipping_city,shipping_province,sales_amount
937,ON100520,15/07/2026,C11491,P7140,T-Shirt,Apparel,NaN,5.0,WELCOME5,5,Debit Card,London,ON,241.44
977,ON100445,06/08/2026,C10901,P2488,Bluetooth Speaker,Electronics,86.45,NaN,SAVE10,10,Debit Card,Waterloo,ON,77.81
1003,ON100587,18/08/2026,C12462,P3140,Bluetooth Speaker,Electronics,21.55,5.0,SAVE10,10,Debit Card,NaN,ON,96.98
1014,ON100504,27/08/2026,C12088,P6690,Granola Bars,Grocery,NaN,2.0,NaN,0,Debit Card,Guelph,ON,14.32


In [77]:
# Show examples of non-positive prices
sales_data[
    pd.to_numeric(
        sales_data["price"],
        errors="coerce"
    ) <= 0
].head(5)

,order_id,date,customer_id,product_id,product,product_category,price,quantity,coupon_code,discount_pct,payment_method,shipping_city,shipping_province,sales_amount


## Step 7 — Cleaning Rules

The cleaning rules are implemented inside the `clean()` method of the `LoadSales` class. The rules standardize text, convert dates and numeric columns, remove rows with missing required fields, remove invalid prices and quantities, fix discount values, and remove exact duplicate rows.

In [78]:
# Record the number of rows before cleaning
rows_before = len(sales_data)

# Execute the cleaning rules inside the class
cleaned_sales = loader.clean()

# Record the number of rows after cleaning
rows_after = len(cleaned_sales)

print(f"Rows before cleaning: {rows_before}")
print(f"Rows after cleaning: {rows_after}")
print(f"Rows removed: {rows_before - rows_after}")

print(
    "Missing values after cleaning:",
    int(cleaned_sales.isna().sum().sum())
)

print(
    "Duplicate rows after cleaning:",
    int(cleaned_sales.duplicated().sum())
)

Rows before cleaning: 1020
Rows after cleaning: 384
Rows removed: 636
Missing values after cleaning: 0
Duplicate rows after cleaning: 0


## Step 8 — Transformations

The `coupon_code` field is transformed into a numeric `coupon_discount_pct` field by extracting the numeric portion of the coupon code. A value of 0 is used when no numeric discount is present. Gross and net revenue are also calculated from the cleaned price, quantity, and discount values.

In [79]:
# Convert the coupon code into a numeric discount percentage
cleaned_sales["coupon_discount_pct"] = (
    cleaned_sales["coupon_code"]
    .astype("string")
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .fillna(0)
    .astype(float)
)

# Calculate gross revenue
cleaned_sales["gross_revenue"] = (
    cleaned_sales["price"]
    * cleaned_sales["quantity"]
)

# Calculate net revenue after the existing discount percentage
cleaned_sales["net_revenue"] = (
    cleaned_sales["gross_revenue"]
    * (1 - cleaned_sales["discount_pct"] / 100)
)

cleaned_sales[
    [
        "coupon_code",
        "coupon_discount_pct",
        "price",
        "quantity",
        "gross_revenue",
        "net_revenue"
    ]
].head(10)

,coupon_code,coupon_discount_pct,price,quantity,gross_revenue,net_revenue
0,NO_COUPON,0.0,63.41,6.0,380.46,380.4600
1,WELCOME5,5.0,55.39,5.0,276.95,263.1025
2,NO_COUPON,0.0,81.91,3.0,245.73,245.7300
3,NO_COUPON,0.0,93.35,3.0,280.05,280.0500
4,NO_COUPON,0.0,14.40,3.0,43.20,43.2000
5,NO_COUPON,0.0,9.20,2.0,18.40,18.4000
6,WELCOME5,5.0,206.19,3.0,618.57,587.6415
7,SAVE10,10.0,16.79,2.0,33.58,30.2220
8,FREESHIP,0.0,14.66,1.0,14.66,14.6600
9,NO_COUPON,0.0,153.10,1.0,153.10,153.1000


## Step 9 — Feature Engineering

A new `days_since_purchase` feature is created to show how old each transaction is relative to the latest purchase date in the cleaned dataset. Using the latest dataset date as the reference makes the calculation reproducible.

In [80]:
# Use the latest purchase date as the reference date
reference_date = cleaned_sales["date"].max().normalize()

# Calculate days since purchase
cleaned_sales["days_since_purchase"] = (
    reference_date
    - cleaned_sales["date"].dt.normalize()
).dt.days

print("Reference date:", reference_date)

cleaned_sales[
    [
        "date",
        "days_since_purchase"
    ]
].head(10)

Reference date: 2026-12-08 00:00:00


,date,days_since_purchase
0,2025-03-01,647
1,2025-03-01,647
2,2025-04-01,616
3,2025-04-01,616
4,2025-04-01,616
5,2025-04-01,616
6,2025-05-01,586
7,2025-05-01,586
8,2025-05-01,586
9,2025-07-01,525


## Step 10 — Mini-Aggregation

The cleaned dataset is grouped by `shipping_city` to calculate total net revenue for each city. The result is also converted into a dictionary to demonstrate another Python data structure.

In [81]:
# Calculate revenue by shipping city
revenue_by_city = (
    cleaned_sales
    .groupby("shipping_city")["net_revenue"]
    .sum()
    .sort_values(ascending=False)
)

# Convert the result to a dictionary
revenue_by_city_dict = revenue_by_city.to_dict()

# Display the top 10 cities
revenue_by_city.head(10)

shipping_city
Ottawa         7199.1425
Guelph         6927.3350
Cambridge      5152.9670
Scarborough    4594.2920
Mississauga    4590.5835
Brampton       4514.8745
Hamilton       4504.7820
Windsor        4328.9785
Kitchener      4298.6545
London         4089.8055
Name: net_revenue, dtype: float64

In [82]:
top_city = revenue_by_city.index[0]
top_city_revenue = revenue_by_city.iloc[0]

total_revenue = revenue_by_city.sum()

top_city_share = (
    top_city_revenue / total_revenue * 100
)

print(
    f"The highest net revenue came from {top_city}, "
    f"with ${top_city_revenue:,.2f}, representing "
    f"{top_city_share:.2f}% of total net revenue."
)

The highest net revenue came from Ottawa, with $7,199.14, representing 10.96% of total net revenue.


## Step 11 — Serialization Checkpoint

The cleaned dataset is serialized into both CSV and JSON formats. JSON is stored as a list of records so that each transaction remains a separate object.

In [83]:
import json

# Make sure the output folder exists
OUTPUT_DIR.mkdir(exist_ok=True)

# Output file paths
cleaned_csv_path = OUTPUT_DIR / "cleaned_sales.csv"
cleaned_json_path = OUTPUT_DIR / "cleaned_sales.json"

# Save cleaned CSV
cleaned_sales.to_csv(
    cleaned_csv_path,
    index=False
)

# Save cleaned JSON
cleaned_sales.to_json(
    cleaned_json_path,
    orient="records",
    date_format="iso",
    indent=2
)

print("Saved:")
print(cleaned_csv_path)
print(cleaned_json_path)

Saved:
d:\Career\Conestoga\MLP\week3\DataEngineering_Lab1\output\cleaned_sales.csv
d:\Career\Conestoga\MLP\week3\DataEngineering_Lab1\output\cleaned_sales.json


In [84]:
# Read the JSON file back into Python
with open(cleaned_json_path, "r", encoding="utf-8") as file:
    saved_records = json.load(file)

print("Records written to JSON:", len(saved_records))
print("Records in cleaned DataFrame:", len(cleaned_sales))
print("Serialization check:", len(saved_records) == len(cleaned_sales))

Records written to JSON: 384
Records in cleaned DataFrame: 384
Serialization check: True


## Soft Interview Reflection

Functions have helped me organize the data workflow into clear, reusable steps. Instead of placing all logic in one long block, each function can focus on a specific responsibility, such as loading data, cleaning values, or transforming columns. This makes the code easier to read, test, debug, and reuse in other projects. Functions also improve consistency because the same operation can be called whenever it is needed. In an interview, I could explain that using functions demonstrates structured problem-solving and makes a data pipeline easier for another developer to understand and maintain.

# Data Dictionary

The Data Dictionary combines definitions from the primary sales dataset and the secondary product metadata source. New fields created during cleaning, transformation, and feature engineering are also included.

In [85]:
# Load the primary and secondary datasets

primary_df = pd.read_csv(
    DATA_DIR / "retail_sales_ontario_synthetic.csv"
)

secondary_df = pd.read_csv(
    DATA_DIR / "product_metadata.csv"
)

print("Primary CSV columns:")
print(primary_df.columns.tolist())

print("\nSecondary metadata columns:")
print(secondary_df.columns.tolist())

Primary CSV columns:
['order_id', 'date', 'customer_id', 'product_id', 'product', 'product_category', 'price', 'quantity', 'coupon_code', 'discount_pct', 'payment_method', 'shipping_city', 'shipping_province', 'sales_amount']

Secondary metadata columns:
['field', 'type', 'description', 'source', 'source_url']


## 1. Primary CSV Field Definitions

The following definitions describe the fields originally provided in the primary e-commerce transaction dataset.


In [86]:
# Define the fields from the primary sales CSV

primary_dictionary = pd.DataFrame([
    {
        "Field": "order_id",
        "Type": "string",
        "Description": "Unique identifier for a sales transaction.",
        "Source": "Primary CSV"
    },
    {
        "Field": "date",
        "Type": "datetime",
        "Description": "Date on which the sales transaction occurred.",
        "Source": "Primary CSV"
    },
    {
        "Field": "customer_id",
        "Type": "string",
        "Description": "Identifier for the customer.",
        "Source": "Primary CSV"
    },
    {
        "Field": "product_id",
        "Type": "string",
        "Description": "Identifier for the purchased product.",
        "Source": "Primary CSV"
    },
    {
        "Field": "product",
        "Type": "string",
        "Description": "Name of the purchased product.",
        "Source": "Primary CSV"
    },
    {
        "Field": "product_category",
        "Type": "string",
        "Description": "Category of the purchased product.",
        "Source": "Primary CSV"
    },
    {
        "Field": "price",
        "Type": "float",
        "Description": "Price per unit of the purchased product.",
        "Source": "Primary CSV"
    },
    {
        "Field": "quantity",
        "Type": "numeric",
        "Description": "Number of units purchased.",
        "Source": "Primary CSV"
    },
    {
        "Field": "coupon_code",
        "Type": "string",
        "Description": "Coupon or promotional code associated with the transaction.",
        "Source": "Primary CSV"
    },
    {
        "Field": "discount_pct",
        "Type": "float",
        "Description": "Percentage discount applied to the transaction.",
        "Source": "Primary CSV"
    },
    {
        "Field": "payment_method",
        "Type": "string",
        "Description": "Payment method used for the purchase.",
        "Source": "Primary CSV"
    },
    {
        "Field": "shipping_city",
        "Type": "string",
        "Description": "City where the order was shipped.",
        "Source": "Primary CSV"
    },
    {
        "Field": "shipping_province",
        "Type": "string",
        "Description": "Province where the order was shipped.",
        "Source": "Primary CSV"
    },
    {
        "Field": "sales_amount",
        "Type": "float",
        "Description": "Sales amount recorded for the transaction.",
        "Source": "Primary CSV"
    }
])

primary_dictionary

,Field,Type,Description,Source
0,order_id,string,Unique identifier for a sales transaction.,Primary CSV
1,date,datetime,Date on which the sales transaction occurred.,Primary CSV
2,customer_id,string,Identifier for the customer.,Primary CSV
3,product_id,string,Identifier for the purchased product.,Primary CSV
4,product,string,Name of the purchased product.,Primary CSV
5,product_category,string,Category of the purchased product.,Primary CSV
6,price,float,Price per unit of the purchased product.,Primary CSV
7,quantity,numeric,Number of units purchased.,Primary CSV
8,coupon_code,string,Coupon or promotional code associated with the...,Primary CSV
9,discount_pct,float,Percentage discount applied to the transaction.,Primary CSV


In [87]:
# Select the useful definition columns from the secondary metadata file

secondary_dictionary = secondary_df[
    ["field", "type", "description", "source"]
].copy()

# Rename columns so they match the primary dictionary

secondary_dictionary = secondary_dictionary.rename(
    columns={
        "field": "Field",
        "type": "Type",
        "description": "Description",
        "source": "Source"
    }
)

secondary_dictionary

,Field,Type,Description,Source
0,id,integer,Unique product identifier.,DummyJSON Products API
1,title,string,Product name or title.,DummyJSON Products API
2,category,string,Product category.,DummyJSON Products API
3,price,float,Product price.,DummyJSON Products API
4,discountPercentage,float,Discount percentage associated with the product.,DummyJSON Products API
5,rating,float,Product rating value.,DummyJSON Products API
6,stock,integer,Available stock quantity shown for the product.,DummyJSON Products API
7,brand,string,Product brand.,DummyJSON Products API
8,sku,string,Stock keeping unit identifier.,DummyJSON Products API
9,weight,float,Product weight value.,DummyJSON Products API


## 3. New Fields Created During the Data-Engineering Process

The following fields were not part of the original primary CSV. They were created during the transformation and feature-engineering steps of this lab.


In [88]:
# Define the new columns created during the lab

new_columns = pd.DataFrame([
    {
        "Field": "coupon_discount_pct",
        "Type": "float",
        "Description": "Numeric discount percentage extracted from the coupon_code.",
        "Source": "Created from coupon_code"
    },
    {
        "Field": "gross_revenue",
        "Type": "float",
        "Description": "Revenue calculated by multiplying price by quantity.",
        "Source": "Calculated: price × quantity"
    },
    {
        "Field": "net_revenue",
        "Type": "float",
        "Description": "Revenue after applying the transaction discount.",
        "Source": "Calculated from gross_revenue and discount_pct"
    },
    {
        "Field": "days_since_purchase",
        "Type": "integer",
        "Description": "Number of days between the purchase date and the latest purchase date in the cleaned dataset.",
        "Source": "Feature engineered from date"
    }
])

new_columns

,Field,Type,Description,Source
0,coupon_discount_pct,float,Numeric discount percentage extracted from the...,Created from coupon_code
1,gross_revenue,float,Revenue calculated by multiplying price by qua...,Calculated: price × quantity
2,net_revenue,float,Revenue after applying the transaction discount.,Calculated from gross_revenue and discount_pct
3,days_since_purchase,integer,Number of days between the purchase date and t...,Feature engineered from date


## 4. Merge All Field Definitions

The primary CSV definitions, secondary metadata definitions, and newly created fields are combined into one Data Dictionary. Duplicate fields from the secondary source are removed so that each field appears only once in the final table.


In [89]:
# Combine the primary definitions, secondary metadata,
# and new fields into one Data Dictionary

data_dictionary = pd.concat(
    [
        primary_dictionary,
        secondary_dictionary,
        new_columns
    ],
    ignore_index=True
)

# Remove duplicate field names.
# The first definition is kept, which preserves the
# primary transaction definition when a field exists
# in both sources.

data_dictionary = (
    data_dictionary
    .drop_duplicates(subset="Field", keep="first")
    .sort_values("Field")
    .reset_index(drop=True)
)

data_dictionary

,Field,Type,Description,Source
0,brand,string,Product brand.,DummyJSON Products API
1,category,string,Product category.,DummyJSON Products API
2,coupon_code,string,Coupon or promotional code associated with the...,Primary CSV
3,coupon_discount_pct,float,Numeric discount percentage extracted from the...,Created from coupon_code
4,customer_id,string,Identifier for the customer.,Primary CSV
5,date,datetime,Date on which the sales transaction occurred.,Primary CSV
6,days_since_purchase,integer,Number of days between the purchase date and t...,Feature engineered from date
7,discountPercentage,float,Discount percentage associated with the product.,DummyJSON Products API
8,discount_pct,float,Percentage discount applied to the transaction.,Primary CSV
9,gross_revenue,float,Revenue calculated by multiplying price by qua...,Calculated: price × quantity


## 5. Final Data Dictionary

The final Data Dictionary contains the original transaction fields, fields documented by the secondary metadata source, and the new fields created during the lab.


In [94]:
# Display the final Data Dictionary as a Markdown table

print(
    data_dictionary.to_markdown(index=False)
)

| Field               | Type     | Description                                                                                   | Source                                         |
|:--------------------|:---------|:----------------------------------------------------------------------------------------------|:-----------------------------------------------|
| brand               | string   | Product brand.                                                                                | DummyJSON Products API                         |
| category            | string   | Product category.                                                                             | DummyJSON Products API                         |
| coupon_code         | string   | Coupon or promotional code associated with the transaction.                                   | Primary CSV                                    |
| coupon_discount_pct | float    | Numeric discount percentage extracted from the coupon_code.      

## 6. How the Fields Were Created

The original transaction fields came from the primary CSV dataset. The secondary metadata fields came from the separate product metadata source. The `coupon_discount_pct` field was created by extracting the numeric part of `coupon_code`. `gross_revenue` was calculated using `price × quantity`. `net_revenue` was calculated by applying the transaction discount to gross revenue. `days_since_purchase` was created by calculating the number of days between each purchase date and the latest purchase date in the cleaned dataset. The primary sales dataset is synthetic, while the new analytical fields are calculated or feature-engineered from the existing data.
